In [2]:
import torch
from torch import nn
import torch.nn.functional as F
from dataclasses import dataclass

In [3]:
@dataclass
class GLM4MoeConfig:
    vocab_size: int = 151552
    hidden_size: int = 5120
    num_hidden_layers: int = 92
    rms_norm_eps: float = 1e-5
    num_attention_heads: int = 96
    num_key_value_heads: int = 8
    head_dim: int = 128
    attention_bias: bool = True
    use_qk_norm: bool = True
    rope_theta: float = 1_000_000
    partial_rotary_factor: float = 0.5
    intermediate_size: int = 12288
    hidden_act: type[nn.Module] = nn.SiLU
    first_k_dense_replace: int = 3
    n_routed_experts: int = 160
    num_experts_per_tok: int = 8
    moe_intermediate_size: int = 1536
    n_shared_experts: int = 1
    routed_scaling_factor: float = 2.5
    norm_topk_prob: bool = True
    n_group: int = 1
    topk_group: int = 1

In [4]:
tiny = GLM4MoeConfig(
    vocab_size=256,
    hidden_size=64,
    num_hidden_layers=6,  # Dense 3 + MoE 3
    num_attention_heads=4,
    num_key_value_heads=2,  # GQA 2:1
    head_dim=32,
    intermediate_size=128,
    moe_intermediate_size=48,
    n_routed_experts=8,
    num_experts_per_tok=2,
    routed_scaling_factor=2.5,
    rope_theta=10000,
)

In [5]:
class KVCache:
    def __init__(self):
        self.cache: list[tuple[torch.Tensor, torch.Tensor]] = []

    def update(self, layer_idx: int, k: torch.Tensor, v: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        if layer_idx < len(self.cache):
            prev_k, prev_v = self.cache[layer_idx]
            k = torch.cat([prev_k, k], dim=1)
            v = torch.cat([prev_v, v], dim=1)
            self.cache[layer_idx] = (k, v)
        else:
            self.cache.append((k, v))
        return k, v

    @property
    def seq_length(self) -> int:
        # (B, L)
        return self.cache[0][0].shape[2] if self.cache else 0

In [6]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor):
        input_dtype = x.dtype
        x = x.to(torch.float32)
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * x.to(input_dtype)

In [43]:
class RotaryEmbedding(nn.Module):
    def __init__(self, config: GLM4MoeConfig):
        super().__init__()
        dim = int(config.head_dim * config.partial_rotary_factor)
        inv_freq = 1.0 / (config.rope_theta ** (torch.arange(0, dim, 2).to(torch.float32) / dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x: torch.Tensor, position_ids: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # (B, dim/2, 1)
        inv_freq = self.inv_freq[None, :, None].expand(position_ids.shape[0], -1, 1)
        # (B, 1, L)
        pos = position_ids[:, None, :].to(torch.float32)
        # (B, L, dim/2)
        freqs = (inv_freq.to(torch.float32) @ pos).transpose(1, 2)
        # (B, L, dim/2 * 2)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos().to(x.dtype)

In [44]:
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

In [30]:
position_ids = torch.arange(8)[None, :].expand(2, -1)

In [31]:
position_ids.shape

torch.Size([2, 8])

In [32]:
pos = position_ids[:, None, :].to(torch.float32)
pos.shape

torch.Size([2, 1, 8])

In [33]:
inv_freq = 1.0 / (1000 ** (torch.arange(0, 8, 2).to(torch.float32) / 8))

In [34]:
inv_freq

tensor([1.0000, 0.1778, 0.0316, 0.0056])

In [35]:
inv_freq.shape

torch.Size([4])

In [36]:
_inv_freq = inv_freq[None, :, None].expand(position_ids.shape[0], -1, 1)

In [37]:
_inv_freq.shape

torch.Size([2, 4, 1])

In [38]:
(_inv_freq @ pos).shape

torch.Size([2, 4, 8])

In [39]:
freqs = (_inv_freq @ pos).transpose(1, 2)

In [40]:
freqs

tensor([[[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [1.0000e+00, 1.7783e-01, 3.1623e-02, 5.6234e-03],
         [2.0000e+00, 3.5566e-01, 6.3246e-02, 1.1247e-02],
         [3.0000e+00, 5.3348e-01, 9.4868e-02, 1.6870e-02],
         [4.0000e+00, 7.1131e-01, 1.2649e-01, 2.2494e-02],
         [5.0000e+00, 8.8914e-01, 1.5811e-01, 2.8117e-02],
         [6.0000e+00, 1.0670e+00, 1.8974e-01, 3.3740e-02],
         [7.0000e+00, 1.2448e+00, 2.2136e-01, 3.9364e-02]],

        [[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [1.0000e+00, 1.7783e-01, 3.1623e-02, 5.6234e-03],
         [2.0000e+00, 3.5566e-01, 6.3246e-02, 1.1247e-02],
         [3.0000e+00, 5.3348e-01, 9.4868e-02, 1.6870e-02],
         [4.0000e+00, 7.1131e-01, 1.2649e-01, 2.2494e-02],
         [5.0000e+00, 8.8914e-01, 1.5811e-01, 2.8117e-02],
         [6.0000e+00, 1.0670e+00, 1.8974e-01, 3.3740e-02],
         [7.0000e+00, 1.2448e+00, 2.2136e-01, 3.9364e-02]]])

In [41]:
emb = torch.cat((freqs, freqs), dim=-1)
emb

tensor([[[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
          0.0000e+00, 0.0000e+00, 0.0000e+00],
         [1.0000e+00, 1.7783e-01, 3.1623e-02, 5.6234e-03, 1.0000e+00,
          1.7783e-01, 3.1623e-02, 5.6234e-03],
         [2.0000e+00, 3.5566e-01, 6.3246e-02, 1.1247e-02, 2.0000e+00,
          3.5566e-01, 6.3246e-02, 1.1247e-02],
         [3.0000e+00, 5.3348e-01, 9.4868e-02, 1.6870e-02, 3.0000e+00,
          5.3348e-01, 9.4868e-02, 1.6870e-02],
         [4.0000e+00, 7.1131e-01, 1.2649e-01, 2.2494e-02, 4.0000e+00,
          7.1131e-01, 1.2649e-01, 2.2494e-02],
         [5.0000e+00, 8.8914e-01, 1.5811e-01, 2.8117e-02, 5.0000e+00,
          8.8914e-01, 1.5811e-01, 2.8117e-02],
         [6.0000e+00, 1.0670e+00, 1.8974e-01, 3.3740e-02, 6.0000e+00,
          1.0670e+00, 1.8974e-01, 3.3740e-02],
         [7.0000e+00, 1.2448e+00, 2.2136e-01, 3.9364e-02, 7.0000e+00,
          1.2448e+00, 2.2136e-01, 3.9364e-02]],

        [[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.00

In [42]:
emb.cos()

tensor([[[ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
           1.0000],
         [ 0.5403,  0.9842,  0.9995,  1.0000,  0.5403,  0.9842,  0.9995,
           1.0000],
         [-0.4161,  0.9374,  0.9980,  0.9999, -0.4161,  0.9374,  0.9980,
           0.9999],
         [-0.9900,  0.8610,  0.9955,  0.9999, -0.9900,  0.8610,  0.9955,
           0.9999],
         [-0.6536,  0.7575,  0.9920,  0.9997, -0.6536,  0.7575,  0.9920,
           0.9997],
         [ 0.2837,  0.6301,  0.9875,  0.9996,  0.2837,  0.6301,  0.9875,
           0.9996],
         [ 0.9602,  0.4828,  0.9821,  0.9994,  0.9602,  0.4828,  0.9821,
           0.9994],
         [ 0.7539,  0.3203,  0.9756,  0.9992,  0.7539,  0.3203,  0.9756,
           0.9992]],

        [[ 1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
           1.0000],
         [ 0.5403,  0.9842,  0.9995,  1.0000,  0.5403,  0.9842,  0.9995,
           1.0000],
         [-0.4161,  0.9374,  0.9980,  0.9999, -0.4161,  0.9374,  0.9